In [ ]:
import numpy as np

def iterative_math(condition=1, starting_param_list=None, model=2, gens=15, print_gens=None):
    """
    Calculates the expected variance and covariance iteratively based on a given condition,
    printing each matrix at each generation.

    Args:
        condition (int): The condition number (1-5) to use for parameter selection.
        starting_param_list (dict): A dictionary of starting parameters. If None, uses default values.
        model (int): The model number to use (only model 2 is implemented here).
        gens (int): The number of generations to simulate.
        print_gens (list): List of generation numbers to print. If None, prints all generations.
    """
    if starting_param_list is None:
        starting_param_list = {
            'vg1': .75,
            'vg2': 1,
            'am11': 0,
            'am12': 0,
            'am21': 0,
            'am22': 0.43,
            'f11': 0,
            'f12': 0.0,
            'f21': 0.0,
            'f22': 0,
            'Nfam': 1.6e5,
            'rg': 0.5,
            're': 0,
            'prop.h2.latent1': .56/0.75,
            'prop.h2.latent2': 0.65/0.8
        }

    # --- Parameter setup for the given condition ---
    idx = condition - 1
    vg1 = starting_param_list['vg1'][idx] if isinstance(starting_param_list['vg1'], np.ndarray) else starting_param_list['vg1']
    vg2 = starting_param_list['vg2'][idx] if isinstance(starting_param_list['vg2'], np.ndarray) else starting_param_list['vg2']
    rg = starting_param_list['rg'][idx] if isinstance(starting_param_list['rg'], np.ndarray) else starting_param_list['rg']
    prop_h2_latent1 = starting_param_list['prop.h2.latent1'][idx] if isinstance(starting_param_list['prop.h2.latent1'], np.ndarray) else starting_param_list['prop.h2.latent1']
    prop_h2_latent2 = starting_param_list['prop.h2.latent2'][idx] if isinstance(starting_param_list['prop.h2.latent2'], np.ndarray) else starting_param_list['prop.h2.latent2']
    am11 = starting_param_list['am11'][idx] if isinstance(starting_param_list['am11'], np.ndarray) else starting_param_list['am11']
    am12 = starting_param_list['am12'][idx] if isinstance(starting_param_list['am12'], np.ndarray) else starting_param_list['am12']
    am21 = starting_param_list['am21'][idx] if isinstance(starting_param_list['am21'], np.ndarray) else starting_param_list['am21']
    am22 = starting_param_list['am22'][idx] if isinstance(starting_param_list['am22'], np.ndarray) else starting_param_list['am22']
    f11 = starting_param_list['f11'][idx] if isinstance(starting_param_list['f11'], np.ndarray) else starting_param_list['f11']
    f12 = starting_param_list['f12'][idx] if isinstance(starting_param_list['f12'], np.ndarray) else starting_param_list['f12']
    f21 = starting_param_list['f21'][idx] if isinstance(starting_param_list['f21'], np.ndarray) else starting_param_list['f21']
    f22 = starting_param_list['f22'][idx] if isinstance(starting_param_list['f22'], np.ndarray) else starting_param_list['f22']
    re = starting_param_list['re'][idx] if isinstance(starting_param_list['re'], np.ndarray) else starting_param_list['re']

    # --- Implied variables (t0) ---
    k2_matrix = np.array([[1, rg], [rg, 1]])

    vg_obs1 = vg1 * (1 - prop_h2_latent1)
    vg_obs2 = vg2 * (1 - prop_h2_latent2)
    d11 = np.sqrt(vg_obs1)
    d21 = 0
    d22 = np.sqrt(vg_obs2 - d21**2)
    delta_mat = np.array([[d11, 0], [d21, d22]])

    vg_lat1 = vg1 * prop_h2_latent1
    vg_lat2 = vg2 * prop_h2_latent2
    a11 = np.sqrt(vg_lat1)
    a21 = 0
    a22 = np.sqrt(vg_lat2 - a21**2)
    a_mat = np.array([[a11, 0], [a21, a22]])

    covg_mat = (delta_mat @ k2_matrix @ delta_mat.T) + (a_mat @ k2_matrix @ a_mat.T)

    ve1 = 1 - vg1
    ve2 = 1 - vg2
    cove = re * np.sqrt(ve1 * ve2)
    cove_mat = np.array([[ve1, cove], [cove, ve2]])

    COVY = covg_mat + cove_mat

    mate_cor_mat = np.array([[am11, am12], [am21, am22]])
    f_mat = np.array([[f11, f12], [f21, f22]])
    covf_mat = 2 * (f_mat @ COVY @ f_mat.T)

    # --- Initialize parameters for iteration ---
    a_t0 = a_mat
    delta_t0 = delta_mat
    j_t0 = k2_matrix * 0.5
    k_t0 = k2_matrix * 0.5
    f_t0 = f_mat
    rmate_t0 = mate_cor_mat
    covE_t0 = cove_mat

    # Lists to store matrices at each generation
    exp_VY, exp_VF, exp_mu, mate_cov = ([None] * gens for _ in range(4))
    exp_gc, exp_hc, exp_ic, exp_w, exp_v = ([None] * gens for _ in range(5))
    exp_Omega, exp_Gamma, exp_itlo, exp_itol = ([None] * gens for _ in range(4))
    exp_VGO, exp_heritability, exp_cor_matpgs, exp_VGL, exp_COVLO = ([None] * gens for _ in range(5))

    # Initialize matrices at t=0 (index 0)
    exp_gc[0] = exp_hc[0] = exp_ic[0] = exp_w[0] = exp_v[0] = np.zeros((2, 2))
    exp_VY[0] = COVY
    exp_VF[0] = covf_mat
    exp_mu[0] = np.linalg.inv(COVY) @ rmate_t0 @ np.linalg.inv(COVY.T)
    mate_cov[0] = COVY @ exp_mu[0] @ COVY

    print(f"=========================================")
    print(f"STARTING SIMULATION")
    print(f"=========================================\n")
    
    if model == 2:
        for it in range(1, gens):
            it_prev = it - 1
            
            # Determine if we should print this generation
            should_print = print_gens is None or it in print_gens
            
            if should_print & False:
                print(f"\n---------- Generation {it} ----------")

            exp_Omega[it] = (2 * delta_t0 @ exp_gc[it_prev] + delta_t0 @ k_t0 +
                             0.5 * exp_w[it_prev] + 2 * a_t0 @ exp_ic[it_prev])
            if should_print & False:
                print(f"Omega ({it}):\n{exp_Omega[it]}")

            exp_Gamma[it] = (2 * a_t0 @ exp_hc[it_prev] + 2 * delta_t0 @ exp_ic[it_prev].T +
                             a_t0 @ j_t0 + 0.5 * exp_v[it_prev])
            if should_print & False:
                print(f"Gamma ({it}):\n{exp_Gamma[it]}")

            exp_VY[it] = (2 * delta_t0 @ exp_Omega[it].T + 2 * a_t0 @ exp_Gamma[it].T +
                          exp_w[it_prev] @ delta_t0.T + exp_v[it_prev] @ a_t0.T +
                          exp_VF[it_prev] + covE_t0)
            if should_print & False:
                print(f"VY ({it}):\n{exp_VY[it]}")
            
            vy_sqrt_diag = np.sqrt(np.diag(np.diag(exp_VY[it])))
            mate_cov[it] = vy_sqrt_diag @ rmate_t0 @ vy_sqrt_diag
            exp_mu[it] = np.linalg.inv(exp_VY[it]) @ mate_cov[it] @ np.linalg.inv(exp_VY[it].T)
            if should_print & False:
                print(f"mu ({it}):\n{exp_mu[it]}")
                print(f"mate_cov ({it}):\n{mate_cov[it]}")
            exp_gt = exp_Omega[it].T @ exp_mu[it] @ exp_Omega[it]
            if should_print & False:
                print(f"gt ({it}):\n{exp_gt}")
            
            exp_gc[it] = 0.5 * (exp_gt + exp_gt.T)
            if should_print & False:
                print(f"gc ({it}):\n{exp_gc[it]}")

            exp_ht = exp_Gamma[it].T @ exp_mu[it] @ exp_Gamma[it]
            if should_print & False:
                print(f"ht ({it}):\n{exp_ht}")
            
            exp_hc[it] = 0.5 * (exp_ht + exp_ht.T)
            if should_print & False:
                print(f"hc ({it}):\n{exp_hc[it]}")

            exp_w[it] = (2 * f_t0 @ exp_Omega[it] +
                         f_t0 @ exp_VY[it] @ exp_mu[it] @ exp_Omega[it] +
                         f_t0 @ exp_VY[it] @ exp_mu[it].T @ exp_Omega[it])
            if should_print & False:
                print(f"w ({it}):\n{exp_w[it]}")

            exp_v[it] = (2 * f_t0 @ exp_Gamma[it] +
                         f_t0 @ exp_VY[it] @ exp_mu[it] @ exp_Gamma[it] +
                         f_t0 @ exp_VY[it] @ exp_mu[it].T @ exp_Gamma[it])
            if should_print & False:
                print(f"v ({it}):\n{exp_v[it]}")

            exp_VF[it] = (2 * f_t0 @ exp_VY[it] @ f_t0.T +
                          f_t0 @ exp_VY[it] @ exp_mu[it] @ exp_VY[it] @ f_t0.T +
                          f_t0 @ exp_VY[it] @ exp_mu[it].T @ exp_VY[it] @ f_t0.T)
            if should_print:
                print(f"VF ({it}):\n{exp_VF[it]}")

            exp_itlo[it] = exp_Gamma[it].T @ exp_mu[it] @ exp_Omega[it]
            exp_itol[it] = exp_Omega[it].T @ exp_mu[it] @ exp_Gamma[it]
            exp_ic[it] = 0.25 * (exp_itol[it] + exp_itol[it].T + exp_itlo[it] + exp_itlo[it].T)
            if should_print:
                print(f"ic ({it}):\n{exp_ic[it]}")

            exp_VGO[it] = (2 * delta_t0 @ k_t0 @ delta_t0.T + 4 * delta_t0 @ exp_gc[it] @ delta_t0.T)
            exp_VGL[it] = (2 * a_t0 @ j_t0 @ a_t0.T + 4 * a_t0 @ exp_hc[it] @ a_t0.T)
            exp_heritability[it] = (exp_VGL[it] + exp_VGO[it] + 8* a_t0 @ exp_ic[it]@delta_t0) / exp_VY[it]
            if should_print:
                print(f"Heritability ({it}):\n{exp_heritability[it]}")
            
            exp_COVLO[it] = (4 * delta_t0 @ exp_ic[it].T @ a_t0.T + 4 * a_t0 @ exp_ic[it] @ delta_t0.T)
            
            exp_cor_matpgs[it] = (exp_Omega[it].T @ exp_mu[it] @ exp_Omega[it] *2 + exp_Omega[it].T @ exp_mu[it].T@ exp_Omega[it]*2)/ (2*k_t0 + 4*exp_gc[it])
            if should_print:
                print(f"Mate Correlation PGS ({it}):\n{exp_cor_matpgs[it]}")
            
            r2pgs_final = exp_VGO[it]/exp_VY[it]
            if should_print:
                print(f"R2 PGS ({it}):\n{r2pgs_final}")
    else:
        print(f"Model {model} is not implemented in this script.")
        return

    # # --- Print final results ---
    # print(f"\n--- Final Results after {gens-1} generations ---")
    # print("True a:\n", a_mat)
    # print("\nTrue delta:\n", delta_mat)
    # print(f"\nr2pgs1: {delta_mat[0, 0]**2:.4f}")
    # print(f"r2pgs2: {delta_mat[1, 1]**2:.4f}")
    # print("-" * 45, "\n")
